# 05. Module B: Final CatBoost Model Evaluation

This notebook provides a comprehensive evaluation of the final CatBoost model for price forecasting. 
We assess performance across multiple metrics, different forecasting horizons (1h-24h), and visualize the model's ability to handle volatility and capture uncertainty.

In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from catboost import CatBoostRegressor, Pool
from sklearn.metrics import accuracy_score, confusion_matrix

# Add src to path
sys.path.append(os.path.abspath("../src"))

%load_ext autoreload
%autoreload 2

from module_b.data_utils import build_features, CONTEXT_LEN, HORIZON
from module_b.train import prepare_catboost_data_local
from evaluation.metrics import calculate_regression_metrics, pinball_loss

plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("viridis")

## 1. Load Data and Model

In [ ]:
checkpoint_dir = Path("../checkpoints/module_b")
model_path = checkpoint_dir / "model.cbm"

if not model_path.exists():
    print(f"ERROR: Model not found at {model_path}. Make sure training has been completed.")
else:
    model = CatBoostRegressor()
    model.load_model(str(model_path))
    print("Model loaded successfully.")

val_raw = pd.read_parquet("../data/splits/val.parquet")
test_raw = pd.read_parquet("../data/splits/test.parquet")

print(f"Validation set size: {len(val_raw)}")
print(f"Test set size: {len(test_raw)}")

## 2. Prediction Generation with Local Descaling

We use the same sliding window approach as in training to ensure the model sees features exactly as they would appear during live inference.

In [ ]:
def generate_predictions(raw_df, split_name, step=HORIZON):
    # 1. Feature Engineering and Local Scaling using the training utility
    X, _ = prepare_catboost_data_local(raw_df, None, split_name, step=step)
    
    if X.empty:
        return pd.DataFrame()
        
    # Get feature names from the model
    feature_names = model.feature_names_
    
    # CatBoost predict (returns Q10, Q50, Q90)
    preds_norm = model.predict(X[feature_names])
    
    # 2. Descaling logic
    feat_df = build_features(raw_df, None, split_name)
    price_arr = feat_df["price"].values
    indices = feat_df.index
    
    all_results = []
    limit = len(feat_df) - CONTEXT_LEN - HORIZON
    
    for i_count, i in enumerate(range(0, limit, step)):
        # Local stats for descaling
        history = price_arr[i : i + CONTEXT_LEN]
        mean_w = np.mean(history)
        std_w = np.std(history) + 1e-6
        last_known_price = history[-1]
        
        # Extract predictions for this 24h window
        start_row = i_count * HORIZON
        end_row = (i_count + 1) * HORIZON
        window_preds_norm = preds_norm[start_row:end_row]
        
        # Descale
        window_preds = window_preds_norm * std_w + mean_w
        
        target_timestamps = indices[i + CONTEXT_LEN : i + CONTEXT_LEN + HORIZON]
        actuals = price_arr[i + CONTEXT_LEN : i + CONTEXT_LEN + HORIZON]
        
        res_df = pd.DataFrame(window_preds, columns=["q10", "q50", "q90"], index=target_timestamps)
        res_df["actual"] = actuals
        res_df["last_known_price"] = last_known_price
        res_df["horizon"] = np.arange(1, HORIZON + 1)
        all_results.append(res_df)
        
    return pd.concat(all_results)

print("Generating predictions for Test set...")
test_preds = generate_predictions(test_raw, "test", step=HORIZON)
print(f"Generated {len(test_preds)} predictions.")

## 3. Global Performance Metrics

In [ ]:
metrics = calculate_regression_metrics(
    test_preds["actual"].values, 
    test_preds["q10"].values, 
    test_preds["q50"].values, 
    test_preds["q90"].values
)

print("Overall Test Metrics:")
for k, v in metrics.items():
    unit = "EUR/MWh" if k in ["MAE", "RMSE"] else ("%" if "Coverage" in k or "MAPE" in k else "")
    print(f"{k:<12}: {v:>8.2f} {unit}")

## 4. Horizon-wise Analysis

Electricity forecasting accuracy typically degrades as the horizon increases. Let's visualize this trend.

In [ ]:
horizon_metrics = test_preds.groupby("horizon").apply(lambda x: pd.Series({
    "MAE": np.mean(np.abs(x["actual"] - x["q50"])),
    "RMSE": np.sqrt(np.mean((x["actual"] - x["q50"])**2)),
    "Pinball_Loss": (pinball_loss(x["actual"], x["q10"], 0.1) + 
                    pinball_loss(x["actual"], x["q50"], 0.5) + 
                    pinball_loss(x["actual"], x["q90"], 0.9)) / 3
}))

fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.plot(horizon_metrics.index, horizon_metrics["MAE"], marker='o', label="MAE", color="tab:blue")
ax1.plot(horizon_metrics.index, horizon_metrics["RMSE"], marker='s', label="RMSE", color="tab:red")
ax1.set_xlabel("Horizon (Hours)")
ax1.set_ylabel("Error (EUR/MWh)")
ax1.legend(loc="upper left")

ax2 = ax1.twinx()
ax2.plot(horizon_metrics.index, horizon_metrics["Pinball_Loss"], marker='x', label="Avg Pinball", color="tab:green", linestyle="--")
ax2.set_ylabel("Avg Pinball Loss (Quantile Score)")
ax2.legend(loc="upper right")

plt.title("Forecasting Error vs. Horizon")
plt.show()

## 5. Visual Sample Forecasts

Let's look at some specific windows in the test set to see how the model handles spikes or quiet periods.

In [ ]:
sample_starts = test_preds.index[::HORIZON*7] # Every week
n_samples = 3

for i in range(n_samples):
    start_time = sample_starts[i]
    end_time = start_time + pd.Timedelta(hours=HORIZON-1)
    subset = test_preds.loc[start_time:end_time]
    
    plt.figure(figsize=(12, 4))
    plt.plot(subset.index, subset["actual"], 'k-', label="Actual", marker='.')
    plt.plot(subset.index, subset["q50"], 'r--', label="Median Forecast")
    plt.fill_between(subset.index, subset["q10"], subset["q90"], color='red', alpha=0.2, label="Q10-Q90 Confidence")
    
    plt.title(f"Forecast Sample starting {start_time}")
    plt.ylabel("EUR/MWh")
    plt.legend()
    plt.show()

## 6. Comparison Across Test Sets (2024 vs 2025)

If the test set spans multiple regimes, we can compare performance.

In [ ]:
test_preds["year"] = test_preds.index.year
yearly_metrics = test_preds.groupby("year").apply(lambda x: pd.Series({
    "MAE": np.mean(np.abs(x["actual"] - x["q50"])),
    "PB50": pinball_loss(x["actual"], x["q50"], 0.5)
}))
print(yearly_metrics)

## 7. Feature Importance

Understanding which features drive the model's price forecasts.

In [ ]:
importances = model.get_feature_importance()
feature_names = model.feature_names_
fi_df = pd.DataFrame({"feature": feature_names, "importance": importances}).sort_values("importance", ascending=False)

plt.figure(figsize=(10, 10))
sns.barplot(x="importance", y="feature", data=fi_df.head(20))
plt.title("Top 20 CatBoost Feature Importances")
plt.show()

## 8. Error Analysis by Hour of Day

Does the model struggle more during specific times of the day?

In [ ]:
test_preds["hour"] = test_preds.index.hour
hourly_mae = test_preds.groupby("hour")["actual"].apply(lambda x: np.mean(np.abs(x - test_preds.loc[x.index, "q50"])))

plt.figure(figsize=(10, 5))
sns.lineplot(x=hourly_mae.index, y=hourly_mae.values, marker='o')
plt.xticks(range(24))
plt.xlabel("Hour of Day")
plt.ylabel("MAE (EUR/MWh)")
plt.title("Mean Absolute Error by Hour of Day")
plt.grid(True)
plt.show()

## 9. Extreme Event Analysis (Spikes)

Evaluating performance on high-price events (top 10%).

In [ ]:
spike_threshold = test_preds["actual"].quantile(0.9)
spikes = test_preds[test_preds["actual"] >= spike_threshold]
spike_mae = np.mean(np.abs(spikes["actual"] - spikes["q50"]))

print(f"Spike Threshold (90th percentile): {spike_threshold:.2f} EUR/MWh")
print(f"MAE on Spikes: {spike_mae:.2f} EUR/MWh")
print(f"Relative Error on Spikes: {(spike_mae / spikes['actual'].mean()) * 100:.2f}%")

# Coverage on spikes
spike_coverage = np.mean((spikes["actual"] >= spikes["q10"]) & (spikes["actual"] <= spikes["q90"])) * 100
print(f"Q10-Q90 Coverage on Spikes: {spike_coverage:.2f}%")

## 10. Directional Accuracy

How often does the model correctly predict if the price will go up or down compared to the last known price?

In [ ]:
def calculate_directional_accuracy(preds_df):
    # Direction of actual change: current actual vs last actual (at horizon 1, this is T-1)
    # For simplicity, let's look at the change relative to the 'price' feature at time T
    # We'll re-fetch the price at T to avoid alignment issues
    
    # Shift the actuals by 1 to get the 'last known price'
    # (Note: this is a simplification for a single horizon)
    results = []
    for h in range(1, HORIZON + 1):
        h_preds = preds_df[preds_df["horizon"] == h].copy()
        # We need the price at T. For horizon h, actual is at T+h.
        # Price at T is h hours before the actual at T+h.
        h_preds["price_at_T"] = h_preds["actual"].shift(h)
        h_preds = h_preds.dropna()
        
        actual_dir = np.sign(h_preds["actual"] - h_preds["price_at_T"])
        pred_dir = np.sign(h_preds["q50"] - h_preds["price_at_T"])
        
        acc = np.mean(actual_dir == pred_dir) * 100
        results.append({"horizon": h, "DA": acc})
        
    return pd.DataFrame(results)

da_df = calculate_directional_accuracy(test_preds)

plt.figure(figsize=(10, 5))
sns.lineplot(x="horizon", y="DA", data=da_df, marker='o')
plt.axhline(50, color='r', linestyle='--', label="Random Guess")
plt.xlabel("Horizon (Hours)")
plt.ylabel("Directional Accuracy (%)")
plt.title("Directional Accuracy vs. Horizon")
plt.ylim(0, 100)
plt.legend()
plt.show()

print(f"Average Directional Accuracy: {da_df['DA'].mean():.2f}%")

## 11. Short-Term Directional Accuracy (Horizon 1)

Evaluating the model's ability to predict price movement (Increase/Decrease) for the immediate next hour (Horizon 1).

In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score

# Filter for horizon 1
h1_preds = test_preds[test_preds['horizon'] == 1].copy()

# Calculate actual and predicted directions relative to last_known_price
h1_preds['actual_dir'] = np.sign(h1_preds['actual'] - h1_preds['last_known_price'])
h1_preds['pred_dir'] = np.sign(h1_preds['q50'] - h1_preds['last_known_price'])

# Directional Accuracy
da_h1 = accuracy_score(h1_preds['actual_dir'], h1_preds['pred_dir'])

print(f"Directional Accuracy (Horizon 1): {da_h1:.2%}")

# Breakdown
correct_inc = ((h1_preds['actual_dir'] == 1) & (h1_preds['pred_dir'] == 1)).sum()
total_inc = (h1_preds['actual_dir'] == 1).sum()
correct_dec = ((h1_preds['actual_dir'] == -1) & (h1_preds['pred_dir'] == -1)).sum()
total_dec = (h1_preds['actual_dir'] == -1).sum()

print(f"Correct Increases: {correct_inc} / {total_inc} ({correct_inc/total_inc:.2%})")
print(f"Correct Decreases: {correct_dec} / {total_dec} ({correct_dec/total_dec:.2%})")

# Confusion Matrix
labels = [-1, 0, 1]
cm = confusion_matrix(h1_preds['actual_dir'], h1_preds['pred_dir'], labels=labels)
cm_df = pd.DataFrame(
    cm, 
    index=['Actual Down', 'Actual Flat', 'Actual Up'], 
    columns=['Pred Down', 'Pred Flat', 'Pred Up']
)

plt.figure(figsize=(8, 6))
sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues')
plt.title("Directional Confusion Matrix (Horizon 1)")
plt.show()

display(cm_df)